# 03. 코딩 에이전트 제안 기반 자동 연구 경로

이 경로는 `research-orchestrator`가 집계 진단을 실행한 뒤 `awaiting_proposal`에서 멈추고, 코딩 에이전트의 `research-proposal` 스킬이 안전한 JSON 후보를 만든 뒤 같은 실행을 `--resume`하는 방식입니다.

```text
diagnosis -> awaiting_proposal -> agent proposal JSON -> AIDM -> verification -> human review
```

에이전트는 runner가 기록한 `proposal-context.json`과 `proposal-catalog.json`만 근거로 사용합니다. catalog 밖의 피처·모델·파라미터, 타깃/실측 입력, 예산 초과 proposal은 AIDM 실행 전에 거부됩니다.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def repo_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('power-forecasting 저장소 안에서 실행하세요.')

REPO_ROOT = repo_root(Path.cwd())
SCRIPTS = REPO_ROOT / '.agents' / 'scripts'
env = dict(os.environ)
env['PATH'] = os.pathsep.join([str(Path(sys.executable).parent), env.get('PATH', '')])
CONFIG_DIR = REPO_ROOT / '.agents' / 'runs' / 'notebook-03-auto'
CONFIG = CONFIG_DIR / 'research-config.json'
OUTPUT = CONFIG_DIR / 'output'
DATASET = REPO_ROOT / 'artifacts' / 'demo' / 'dataset.csv'
assert DATASET.exists(), '먼저 01_legacy_baseline.ipynb를 실행하세요.'
shutil.rmtree(OUTPUT, ignore_errors=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG.write_text(json.dumps({
    'schema_version': '1',
    'run_id': 'notebook-03-auto',
    'dataset_path': '../../../artifacts/demo/dataset.csv',
    'legacy_manifest_path': '../../fixtures/promoted-manifest.json',
    'run_dir': 'output',
    'profiles': ['safe_weather'],
    'max_iterations': 1,
    'fold_count': 5,
    'objective': 'NMAE',
    'minimum_improvement': 0.0,
    'max_plant_regression': 1.0,
    'agent_proposals': True,
}, ensure_ascii=False, indent=2), encoding='utf-8')

first = subprocess.run([str(SCRIPTS / 'run-research-loop.sh'), '--config', str(CONFIG)], cwd=REPO_ROOT, text=True, capture_output=True, env=env)
if first.returncode != 0:
    raise RuntimeError(first.stderr)
state = json.loads((OUTPUT / 'state.json').read_text(encoding='utf-8'))
assert state['status'] == 'awaiting_proposal'
print('코딩 에이전트 proposal handoff 준비:', state['status'])


In [ ]:
iteration_dir = OUTPUT / 'iterations' / '001-safe_weather'
context = json.loads((iteration_dir / 'proposal-context.json').read_text(encoding='utf-8'))
catalog = json.loads((iteration_dir / 'proposal-catalog.json').read_text(encoding='utf-8'))
print('남은 평가 예산:', context['remaining_evaluations'])
print('허용 모델 프로필:', list(catalog['model_recipes']))

# 실제 코딩 에이전트는 research-proposal 스킬로 context/catalog를 읽어 이 파일을 작성합니다.
# 노트북은 재현 가능한 catalog-contained fixture 제안으로 그 handoff를 시연합니다.
proposal = json.loads((REPO_ROOT / '.agents' / 'fixtures' / 'model-search-proposal.json').read_text(encoding='utf-8'))
proposal_path = iteration_dir / 'research-proposal.json'
proposal_path.write_text(json.dumps(proposal, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

resume = subprocess.run([str(SCRIPTS / 'run-research-loop.sh'), '--config', str(CONFIG), '--resume'], cwd=REPO_ROOT, text=True, capture_output=True, env=env)
if resume.returncode != 0:
    raise RuntimeError(resume.stderr)
summary = json.loads((OUTPUT / 'research-summary.json').read_text(encoding='utf-8'))
print('최종 상태:', summary['status'])
print('검증 결과:', summary['verifier']['outcome'])


## 스킬과 사람 검토 경계

| 단계 | 스킬 | 경계 |
| --- | --- | --- |
| 상태 머신과 handoff | `research-orchestrator` | `awaiting_proposal`에서 AIDM 전에 멈춤 |
| 집계 진단 | `research-diagnostic` | 원시 고객 행이나 타깃을 기록하지 않음 |
| 다음 후보 작성 | `research-proposal` | catalog 내 JSON만 작성하고 코드를 만들지 않음 |
| 증적 재검증 | `research-verification` | AIDD, release, 배포를 승인하지 않음 |

`ready_for_human_review`는 사람이 AIDD와 release-gate 증적을 별도로 검토할 수 있다는 뜻일 뿐, 배포·병합·고객 시스템 수정 권한은 부여하지 않습니다.